In [2]:
pip install sqlalchemy pymysql

Defaulting to user installation because normal site-packages is not writeable
  Using cached pymysql-1.2.0-py3-none-any.whl.metadata (4.3 kB)
Using cached pymysql-1.2.0-py3-none-any.whl (45 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
%pip show sqlalchemy
%pip show pymysql

Name: SQLAlchemy
Version: 2.0.38
Summary: Database Abstraction Library
Home-page: https://www.sqlalchemy.org
Author: Mike Bayer
Author-email: mike_mp@zzzcomputing.com
License: MIT
Location: C:\Users\Dell\AppData\Roaming\Python\Python313\site-packages
Requires: greenlet, typing-extensions
Required-by: Flask-SQLAlchemy
Note: you may need to restart the kernel to use updated packages.
Name: PyMySQL
Version: 1.2.0
Summary: Pure Python MySQL Driver
Home-page: 
Author: 
Author-email: Inada Naoki <songofacandy@gmail.com>, Yutaka Matsubara <yutaka.matsubara@gmail.com>
License: 
Location: C:\Users\Dell\AppData\Roaming\Python\Python313\site-packages
Requires: 
Required-by: Flask-MySQL
Note: you may need to restart the kernel to use updated packages.


In [85]:
CREATE DATABASE healthcare_db;

USE healthcare_db;

SyntaxError: invalid syntax (2816257516.py, line 1)

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Pandas version: 2.2.3
NumPy version: 2.2.3


In [55]:
from sqlalchemy import create_engine

username = "root"
password = "YOUR_PASSWORD"
host = "localhost"
port = 3306
database = "healthcare_db"

engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
)

print("SQLAlchemy engine created successfully!")

SQLAlchemy engine created successfully!


In [56]:
from sqlalchemy import text
from sqlalchemy.exc import SQLAlchemyError

try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT 1"))
        print("MySQL connection successful:", result.scalar())
except SQLAlchemyError:
    print("MySQL connection failed. Replace YOUR_PASSWORD in the connection cell with your MySQL password, then rerun this cell.")

MySQL connection failed. Replace YOUR_PASSWORD in the connection cell with your MySQL password, then rerun this cell.


In [57]:
# Step 6.4 — Load dataset

file_path = "../Data/clean/healthcare_clean.csv"

df = pd.read_csv(file_path)



In [58]:
print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully!
Shape: (54966, 17)


,name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results,length_of_stay,age_group
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal,2,19-30
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive,6,61-75
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal,15,76+
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal,30,19-30
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal,20,31-45


In [59]:
# Basic inspection

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

Rows: 54966
Columns: 17

Column Names:
['name', 'age', 'gender', 'blood_type', 'medical_condition', 'date_of_admission', 'doctor', 'hospital', 'insurance_provider', 'billing_amount', 'room_number', 'admission_type', 'discharge_date', 'medication', 'test_results', 'length_of_stay', 'age_group']

Data Types:
name                   object
age                     int64
gender                 object
blood_type             object
medical_condition      object
date_of_admission      object
doctor                 object
hospital               object
insurance_provider     object
billing_amount        float64
room_number             int64
admission_type         object
discharge_date         object
medication             object
test_results           object
length_of_stay          int64
age_group              object
dtype: object


In [60]:
# Step 6.8 - Load Pandas DataFrame into MySQL

from sqlalchemy.exc import SQLAlchemyError

table_name = "health_data"

try:
    df.to_sql(
        table_name,
        con=engine,
        if_exists="replace",
        index=False
    )
    print(f"'{table_name}' table loaded successfully into MySQL!")
except SQLAlchemyError:
    print("MySQL load failed. Replace YOUR_PASSWORD in the connection cell with your MySQL password, then rerun this cell.")

MySQL load failed. Replace YOUR_PASSWORD in the connection cell with your MySQL password, then rerun this cell.


In [61]:
# Verify table and row count

from sqlalchemy.exc import SQLAlchemyError

try:
    with engine.connect() as connection:
        result = connection.execute(
            text("SELECT COUNT(*) FROM health_data")
        )

        print("Rows in MySQL:", result.scalar())
except SQLAlchemyError:
    print("MySQL verification failed. Replace YOUR_PASSWORD in the connection cell with your MySQL password, load the table, then rerun this cell.")

MySQL verification failed. Replace YOUR_PASSWORD in the connection cell with your MySQL password, load the table, then rerun this cell.


In [62]:
from sqlalchemy import text
from sqlalchemy.exc import SQLAlchemyError

query = text("""
SELECT *
FROM health_data
LIMIT 10;
""")

try:
    with engine.connect() as connection:
        result = connection.execute(query)
        rows = result.fetchall()

    pd.DataFrame(rows, columns=result.keys())
except SQLAlchemyError:
    print("MySQL query failed. Replace YOUR_PASSWORD in the connection cell with your MySQL password, load the table, then rerun this cell.")

MySQL query failed. Replace YOUR_PASSWORD in the connection cell with your MySQL password, load the table, then rerun this cell.


In [63]:
query = text("""
SELECT *
FROM health_data
LIMIT 10;
""")

In [64]:
query

In [65]:
from sqlalchemy import text
from sqlalchemy.exc import SQLAlchemyError

query = text("""
DESCRIBE health_data;
""")

try:
    with engine.connect() as connection:
        result = connection.execute(query)
        rows = result.fetchall()
        columns = result.keys()

    pd.DataFrame(rows, columns=columns)
except SQLAlchemyError:
    print("MySQL schema query failed. Replace YOUR_PASSWORD in the connection cell with your MySQL password, load the table, then rerun this cell.")

MySQL schema query failed. Replace YOUR_PASSWORD in the connection cell with your MySQL password, load the table, then rerun this cell.


In [66]:
from sqlalchemy.exc import SQLAlchemyError

query = """
SELECT *
FROM health_data
LIMIT 100;
"""

try:
    df_sql = pd.read_sql(query, con=engine)
    df_sql.head()
except SQLAlchemyError:
    print("MySQL query failed. Replace YOUR_PASSWORD in the connection cell with your MySQL password, load the table, then rerun this cell.")

MySQL query failed. Replace YOUR_PASSWORD in the connection cell with your MySQL password, load the table, then rerun this cell.


In [67]:
import os
from sqlalchemy import create_engine

username = os.getenv("MYSQL_USER", "root")
password = os.getenv("MYSQL_PASSWORD", "YOUR_PASSWORD")
host = os.getenv("MYSQL_HOST", "localhost")
port = int(os.getenv("MYSQL_PORT", 3306))
database = os.getenv("MYSQL_DATABASE", "healthcare_db")

engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
)

print("SQLAlchemy engine created successfully!")

SQLAlchemy engine created successfully!


In [68]:
from sqlalchemy import create_engine

engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}"
    f"@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
)

print("SQLAlchemy engine created successfully!")

SQLAlchemy engine created successfully!


In [69]:
from sqlalchemy import create_engine

engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}"
    f"@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
)

print("SQLAlchemy engine created successfully!")

SQLAlchemy engine created successfully!


In [70]:
from sqlalchemy import create_engine

In [71]:
MYSQL_USER = "root"
MYSQL_PASSWORD = "YOUR_PASSWORD"
MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_DATABASE = "healthcare_db"

In [72]:
engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}"
    f"@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
)

print("SQLAlchemy engine created successfully!")

SQLAlchemy engine created successfully!


In [91]:
from sqlalchemy import text

try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT 1"))
        print("MySQL connection successful! ✅")
        print("Result:", result.scalar())

except Exception as e:    print(e)
    print("MySQL connection failed ❌")

IndentationError: unexpected indent (2075714519.py, line 10)

In [92]:
table_name = "healthcare_clean"

try:
    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="replace",
        index=False
    )
    print(f"Data loaded into MySQL table '{table_name}' successfully! ✅")
except Exception as e:
    print("Data load failed ❌")
    print(e)

Data load failed ❌
(pymysql.err.OperationalError) (1049, "Unknown database 'healthcare_db'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [98]:
import os
from sqlalchemy import create_engine

MYSQL_USER = os.getenv("MYSQL_USER", "root")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD", "sjbarad777")
MYSQL_HOST = os.getenv("MYSQL_HOST", "localhost")
MYSQL_PORT = int(os.getenv("MYSQL_PORT", 3306))
MYSQL_DATABASE = os.getenv("MYSQL_DATABASE", "healthcare_db")

engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}",
    pool_pre_ping=True,
)

print("SQLAlchemy engine created successfully! ✅")

SQLAlchemy engine created successfully! ✅


In [99]:
table_name = "healthcare_records"

try:
    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="replace",
        index=False
    )

    print(f"Data loaded into MySQL table '{table_name}' successfully! ✅")

except Exception as e:
    print("Data load failed ❌")
    print(e)

Data load failed ❌
(pymysql.err.OperationalError) (1049, "Unknown database 'healthcare_db'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [100]:
from sqlalchemy import text

try:
    with engine.connect() as connection:
        tables = connection.execute(text("SHOW TABLES")).fetchall()

    if not tables:
        print("No tables found in the database yet.")
    else:
        df_tables = pd.DataFrame(tables, columns=["Tables_in_healthcare_db"])
        display(df_tables)
except Exception as e:
    print("Could not fetch tables ❌")
    print("Check if MySQL is running, the password is correct, and the database 'healthcare_db' exists.")
    print(e)

Could not fetch tables ❌
Check if MySQL is running, the password is correct, and the database 'healthcare_db' exists.
(pymysql.err.OperationalError) (1049, "Unknown database 'healthcare_db'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [101]:
CREATE DATABASE healthcare_db;
USE healthcare_db;
SHOW DATABASES;

SyntaxError: invalid syntax (181191932.py, line 1)